# บทที่ 4: Stack (กองซ้อน)

> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)
> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*
> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)

---

## 4.1 แนวคิด LIFO

Stack ทำงานแบบ **LIFO** (Last In, First Out) เข้าออกทางเดียวคือ *top*

| Operation | ความหมาย | Complexity |
|---|---|---|
| `push(x)` | ใส่ข้อมูลบนสุด | \(O(1)\) |
| `pop()` | เอาตัวบนสุดออก | \(O(1)\) |
| `peek()/top()` | ดูตัวบนสุด | \(O(1)\) |
| `is_empty()` | ว่างหรือไม่ | \(O(1)\) |

**การใช้งานจริง:** call stack ของฟังก์ชัน, undo/redo, ตรวจวงเล็บ, แปลง infix→postfix, DFS, back button ของเบราว์เซอร์

In [ ]:
# แบบที่ 1: ใช้ Python list (append/pop ท้ายลิสต์ = O(1) amortized)
class ArrayStack:
    def __init__(self):
        self._data = []
    def push(self, x):      self._data.append(x)
    def pop(self):
        if self.is_empty(): raise IndexError('pop from empty stack')
        return self._data.pop()
    def peek(self):
        if self.is_empty(): raise IndexError('peek from empty stack')
        return self._data[-1]
    def is_empty(self):     return len(self._data) == 0
    def __len__(self):      return len(self._data)
    def __repr__(self):     return f"Stack(bottom->top: {self._data})"

s = ArrayStack()
for x in (1, 2, 3):
    s.push(x)
print(s, "| peek =", s.peek(), "| pop =", s.pop(), "->", s)

In [ ]:
# แบบที่ 2: ใช้ linked list (push/pop ที่หัว = O(1) แน่นอน ไม่มีการ resize)
class LinkedStack:
    class _N:
        __slots__ = ('v', 'nxt')
        def __init__(self, v, nxt): self.v, self.nxt = v, nxt
    def __init__(self):
        self._top, self._n = None, 0
    def push(self, x):
        self._top = self._N(x, self._top); self._n += 1
    def pop(self):
        if self._n == 0: raise IndexError('empty')
        v, self._top = self._top.v, self._top.nxt
        self._n -= 1
        return v
    def peek(self):  return self._top.v
    def is_empty(self): return self._n == 0
    def __len__(self): return self._n

ls = LinkedStack()
for c in "ABC": ls.push(c)
print([ls.pop() for _ in range(len(ls))])   # -> C B A (LIFO)

## 4.2 การประยุกต์ใช้ Stack

In [ ]:
# 1) ตรวจวงเล็บสมดุล (balanced parentheses) -> O(n)
def is_balanced(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        elif ch in pairs:
            if not st or st.pop() != pairs[ch]:
                return False
    return not st

for t in ["{[()]}", "([)]", "((("]:
    print(f"{t:>8} -> {is_balanced(t)}")

In [ ]:
# 2) แปลง infix -> postfix (Shunting-yard) และประเมินค่า postfix
PREC = {'+': 1, '-': 1, '*': 2, '/': 2, '^': 3}

def infix_to_postfix(expr):
    out, st = [], []
    for tok in expr.split():
        if tok.isalnum():
            out.append(tok)
        elif tok == '(':
            st.append(tok)
        elif tok == ')':
            while st and st[-1] != '(':
                out.append(st.pop())
            st.pop()
        else:
            # ^ เป็น right-associative จึงใช้ > แทน >=
            while st and st[-1] != '(' and (
                PREC[st[-1]] > PREC[tok] or (PREC[st[-1]] == PREC[tok] and tok != '^')):
                out.append(st.pop())
            st.append(tok)
    while st:
        out.append(st.pop())
    return ' '.join(out)

def eval_postfix(expr):
    st = []
    for tok in expr.split():
        if tok.lstrip('-').isdigit():
            st.append(float(tok))
        else:
            b, a = st.pop(), st.pop()
            st.append({'+': a+b, '-': a-b, '*': a*b, '/': a/b, '^': a**b}[tok])
    return st.pop()

e = "3 + 4 * 2 / ( 1 - 5 ) ^ 2"
p = infix_to_postfix(e)
print("infix  :", e)
print("postfix:", p)
print("ค่า    :", eval_postfix(p))

In [ ]:
# 3) Next Greater Element ด้วย monotonic stack -> O(n)
def next_greater(a):
    res = [-1] * len(a)
    st = []                       # เก็บ index ที่ค่ายังไม่พบตัวใหญ่กว่า
    for i, x in enumerate(a):
        while st and a[st[-1]] < x:
            res[st.pop()] = x
        st.append(i)
    return res

# 4) Stack ที่หาค่าน้อยสุดได้ใน O(1) (min stack)
class MinStack:
    def __init__(self):
        self._d, self._m = [], []     # _m เก็บค่า min ณ แต่ละความลึก
    def push(self, x):
        self._d.append(x)
        self._m.append(x if not self._m else min(x, self._m[-1]))
    def pop(self):
        self._m.pop(); return self._d.pop()
    def get_min(self):
        return self._m[-1]

print("next_greater:", next_greater([4, 5, 2, 25, 7, 8]))
ms = MinStack()
for x in (5, 3, 7, 2):
    ms.push(x)
print("min =", ms.get_min(), "| pop", ms.pop(), "-> min =", ms.get_min())

---
## แบบฝึกหัดบทที่ 4

1. สร้าง stack 2 อันจากอาร์เรย์เดียว (two stacks in one array) โดยใช้พื้นที่ให้คุ้มที่สุด
2. เรียงข้อมูลใน stack โดยใช้ stack ช่วยอีกเพียง 1 อัน
3. หาพื้นที่สี่เหลี่ยมที่ใหญ่ที่สุดใน histogram (largest rectangle) ด้วย \(O(n)\)
4. ตรวจว่าสตริงมีรูปแบบซ้ำแบบ redundant parentheses หรือไม่ เช่น `((a+b))`

### เฉลยบทที่ 4

In [ ]:
# ข้อ 1: ให้ stack1 โตจากซ้าย, stack2 โตจากขวา -> เต็มเมื่อชนกัน
class TwoStacks:
    def __init__(self, n):
        self.a = [None] * n
        self.t1, self.t2 = -1, n
    def push1(self, x):
        if self.t1 + 1 == self.t2: raise OverflowError
        self.t1 += 1; self.a[self.t1] = x
    def push2(self, x):
        if self.t1 + 1 == self.t2: raise OverflowError
        self.t2 -= 1; self.a[self.t2] = x
    def pop1(self):
        v = self.a[self.t1]; self.t1 -= 1; return v
    def pop2(self):
        v = self.a[self.t2]; self.t2 += 1; return v

# ข้อ 2: insertion sort บน stack -> O(n^2)
def sort_stack(st):
    tmp = []
    while st:
        x = st.pop()
        while tmp and tmp[-1] > x:
            st.append(tmp.pop())
        tmp.append(x)
    return tmp                      # เรียงจากน้อย -> มาก (bottom->top)

# ข้อ 3: monotonic increasing stack -> O(n)
def largest_rectangle(h):
    st, best = [], 0
    for i, x in enumerate(h + [0]):        # ต่อ 0 เพื่อบังคับเคลียร์ stack
        while st and h[st[-1]] >= x:
            height = h[st.pop()]
            left = st[-1] + 1 if st else 0
            best = max(best, height * (i - left))
        st.append(i)
    return best

# ข้อ 4
def has_redundant_parens(s):
    st = []
    for ch in s:
        if ch == ')':
            top, found_op = st.pop(), False
            while top != '(':
                if top in '+-*/': found_op = True
                top = st.pop()
            if not found_op:
                return True            # ไม่มี operator ข้างใน -> ซ้ำซ้อน
        else:
            st.append(ch)
    return False

ts = TwoStacks(5); ts.push1('a'); ts.push2('z'); print(ts.a)
print(sort_stack([3, 1, 4, 2]))
print(largest_rectangle([2, 1, 5, 6, 2, 3]))
print(has_redundant_parens("((a+b))"), has_redundant_parens("(a+b)"))